# MACE Foundation Models for Atomistic Simulation

This notebook introduces **machine-learned interatomic potentials (MLIPs)**, focusing on MACE-MP-0 — a universal foundation model for atomistic simulation. We use MACE for fast geometry relaxation before running more expensive DFT calculations.

## Overview
**Questions**
- What are MLIPs and why use them?
- How do I use the MACE foundational model?

**Objectives**
- Use ASE and MACE to conduct geometry relaxation on large unit cells.

**Keypoints**
- MLIPs are about ~1000× faster than DFT for forces and geometry relaxation
- foundational models are trained on DFT data for a large number of elements
- MACE-MP-0 is a widely-used foundational model which has been trained on 89 elements.
- Use MACE-MP-0 for the geometry stage; use GPAW (DFT) for the electronic structure stage
- MACE has no concept of charge state, spin, or electronic structure
- The multi-fidelity principle: match method accuracy to the question being asked

## Lecture Slides

The slides for this tutorial are embedded below.
[📥 Download slides (.pptx)](https://github.com/NU-CEM/Atomistic_Simulation/raw/2026/slides/Tutorial2_MaterialsBasics.pptx) &nbsp;|&nbsp; [Open in full screen](https://livenorthumbriaac-my.sharepoint.com/personal/l_whalley_northumbria_ac_uk/_layouts/15/Doc.aspx?sourcedoc={654dfbf3-9585-4090-894e-5657fb09c234}&amp;action=embedview&amp;wdAr=1.7777777777777777)

<iframe
  src="https://livenorthumbriaac-my.sharepoint.com/personal/l_whalley_northumbria_ac_uk/_layouts/15/Doc.aspx?sourcedoc={654dfbf3-9585-4090-894e-5657fb09c234}&amp;action=embedview&amp;wdAr=1.7777777777777777"
  width="100%"
  height="480"
  frameborder="0"
  allowfullscreen="true">
</iframe>

## Using MACE and ASE

MACE integrates with ASE as a standard calculator. The key function is `mace_mp()` which downloads and caches the pre-trained model automatically.

- `model="medium"` is the recommended default — good balance of speed and accuracy
- `dispersion=False` van der Waals correction (use True for layered materials where this bonding is present)

In [3]:
import mace
from mace.calculators import mace_mp

calc = mace_mp(model="medium", dispersion=False)

ModuleNotFoundError: No module named 'mace_mp'

In [ ]:
from ase.build import bulk
# Build a simple structure and compute energy
diamond = bulk('C', 'diamond', a=3.57)
diamond.calc = calc

energy = diamond.get_potential_energy()
forces = diamond.get_forces()

print(f"Diamond total energy: {energy:.4f} eV")
print(f"Max force (should be ~0 at equilibrium): {np.max(np.abs(forces)):.6f} eV/Å")

## 3. Multi-Fidelity Geometry Relaxation

A key strategy in modern computational materials science is **multi-fidelity optimisation**: use a cheap, approximate method to get close to the minimum, then refine with a more expensive method only when needed.

For defect calculations:
- **MACE** handles the heavy lifting of ionic relaxation — finding approximately correct atomic positions in seconds
- **GPAW** then does a single-point electronic structure calculation on the pre-relaxed geometry

This is valid because:
1. MACE geometry errors are typically <1% in bond lengths — within DFT accuracy anyway
2. The GPAW SCF converges faster from a good starting geometry
3. The electronic structure (DOS, defect levels) is not sensitive to sub-percent geometry errors

The alternative — full DFT relaxation — would take 1-2 hours for a 60-atom supercell on a single core. MACE reduces this to ~30 seconds.


In [ ]:
from ase.optimize import BFGS
from ase.build import bulk, make_supercell
from ase.geometry import get_distances
import time

# Build a slightly distorted diamond supercell
prim = bulk('C', 'diamond', a=3.57)
sc = make_supercell(prim, np.diag([2, 2, 2]))

# Add random displacement to simulate an unrelaxed structure
np.random.seed(42)
sc.positions += np.random.uniform(-0.2, 0.2, sc.positions.shape)

sc.calc = mace_mp(model="medium", dispersion=False, default_dtype="float64")

print(f"Initial max force: {np.max(np.abs(sc.get_forces())):.4f} eV/Å")

t0 = time.time()
opt = BFGS(sc, logfile=None)
opt.run(fmax=0.01)
t1 = time.time()

print(f"Final max force:   {np.max(np.abs(sc.get_forces())):.4f} eV/Å")
print(f"Relaxation time:   {t1-t0:.1f} seconds")
print(f"\nFor comparison, GPAW DFT relaxation of same system: ~15-60 minutes")


## 4. MACE for Defect Structures

MACE is particularly useful for defect pre-relaxation. It captures the local distortion around the defect well, giving a good starting geometry for DFT.


In [ ]:
# Relax an NV centre with MACE
from ase.build import bulk, make_supercell
from ase.geometry import get_distances
from ase.optimize import BFGS
import numpy as np

# Build NV defect
prim = bulk('C', 'diamond', a=3.57)
sc = make_supercell(prim, np.diag([3, 3, 3]))

_, D = get_distances(sc.positions, sc.positions, cell=sc.cell, pbc=True)
np.fill_diagonal(D, np.inf)
vac_idx = int(np.argmin(D[0]))

sym = sc.get_chemical_symbols()
sym[0] = 'N'
sc.set_chemical_symbols(sym)

# Store pristine neighbour positions for comparison
neighbour_indices = np.argsort(D[0])[:4].tolist()
# Remove vacancy index
neighbour_indices = [i for i in neighbour_indices if i != vac_idx][:3]
pristine_positions = sc.positions[neighbour_indices].copy()

del sc[vac_idx]
# Update neighbour indices after deletion
neighbour_indices_updated = [i if i < vac_idx else i-1 for i in neighbour_indices]

# Random displacement
np.random.seed(42)
sc.positions += np.random.uniform(-0.1, 0.1, sc.positions.shape)

sc.calc = mace_mp(model="medium", dispersion=False, default_dtype="float64")
opt = BFGS(sc, logfile=None)
opt.run(fmax=0.01)

# Measure inward relaxation of C neighbours toward vacancy
print("C neighbour displacements after MACE relaxation:")
print("(negative = moved toward vacancy)")
for idx in neighbour_indices_updated:
    relaxed_pos = sc.positions[idx]
    # Approximate: compare to ideal lattice position
    print(f"  Atom {idx}: position {relaxed_pos.round(3)}")

print(f"\nMax residual force: {np.max(np.abs(sc.get_forces())):.4f} eV/Å")
print("\nThis geometry is ready to pass to GPAW for electronic structure.")


## Exercise: MACE equation of state

Use MACE to compute the equation of state for . How does it compare to the value in lab?

## Exercise 8.2

Build a 3×3×1 supercell of hBN and create a **boron vacancy** (V_B). Relax the structure with MACE (fmax = 0.01 eV/Å). By how much do the three nearest N neighbours move outward from the vacancy? Is the relaxed structure still planar?